In [ ]:
from pathlib import Path
import json

def run_reporting_agent(state):

    final_report_path = "artifacts/reporting/final_report.md"
    report_json_path = "artifacts/reporting/reporting_summary.json"
    target_column = (state["target_column"] or state.get("next_input", {})["target_column"])
    task_type = (state["task_type"] or state.get("next_input", {})["task_type"])
    task = f"""
Создай финальный Markdown отчет для ML проекта. Обязательно пиши отчет на русском языке.

Путь к датасету:
{state["dataset_path"]}

Целевая колонка:
{target_column}

Вид задачи:
{task_type}

Суммаризированная информация о данных:
{json.dumps(state.get("data_summary", {}), ensure_ascii=False, indent=2)}

Схема:
{json.dumps(state.get("schema", {}), ensure_ascii=False, indent=2)}

Суммаризированная информация о предобработке данных:
{json.dumps(state.get("data_preparation", {}), ensure_ascii=False, indent=2)}

Суммаризированная информация о создании моделей машинного обучения:
{json.dumps(state.get("modeling_summary", {}), ensure_ascii=False, indent=2)}

Артефакты:
{json.dumps(state.get("artifacts", {}), ensure_ascii=False, indent=2)}

Сохрани финальный отчет в путь:
{final_report_path}

Сохрани компактное JSON summary:
{report_json_path}

Markdown отчет должен включать в себя:
1. Название проекта
2. Бизнес задача
3. Общая информация о данных
4. Целевая колонка и тип ML задачи
5. Группы признаков из схемы
6. Обзор о качестве данных
7. Шаги по подготовке данных
8. Созданные признаки
9. Модели, которые были протестированы
10. Информация о лучшей модели
11. Лучшие метрики
12. Если возможно, то информацию о настройке гиперпараметров
13. Бизнес интерпретация результатов
14. Конечные артефакты

Важные правила:
- Не надо обучать модели.
- Не изменяй1 датасеты.
- Не придумывай сам метрики.
- Используй только артефакты и приложенную информацию о работах агентов .
- Есмли какая-то имеющаяся информация пропущена, то просто не пиши о ней.
- Возвращай только компактный JSON.
- Не нужно возвращать сам отчет Markdown. Он должен быть только сохранен в файл с отчетом.

Верни только JSON:
{{
  "agent": "reporting_agent",
  "status": "success",
  "skipped": false,
  "summary": "Final report was created and saved.",
  "artifacts": {{
    "final_report_path": "{final_report_path}",
    "reporting_summary_path": "{report_json_path}"
  }},
  "next_input": {{}},
  "reason": null
}}
"""

    try:
        result = reporting_agent.invoke({
            "messages": [
                {
                    "role": "user",
                    "content": task,
                }
            ]
        })

        ans = result["messages"][-1].content.strip()
        json_ans = extract_json(ans)
        Path("artifacts/reporting").mkdir(parents=True, exist_ok=True)
        report_path = "artifacts/reporting/reporting_agent_result.json"

        with open(report_path, "w", encoding="utf-8") as f:
            json.dump(json_ans, f, ensure_ascii=False, indent=2)
        new_log = {
            "agent": "reporting_agent",
            "status": "success",
            "skipped": False,
            "summary": "Final report was created and saved.",
            "artifacts": {
                "final_report_path": final_report_path,
                "reporting_summary_path": report_json_path,
                "reporting_stage_path": report_path,
            },
            "next_input": {},
            "reason": None,
        }

        return {
            "current_step": "reporting_agent",
            "artifacts": {
                **state.get("artifacts", {}),
                "final_report_path": final_report_path,
                "reporting_summary_path": report_json_path,
                "reporting_stage_path": report_path,
            },
            "logs": [new_log],
            "next_input": {},
            "next_action": None,
            "orchestration_decision": {},
            "status": "success",
            "error": None,
        }

    except Exception as e:
        new_log = {
            "agent": "reporting_agent",
            "status": "failed",
            "skipped": False,
            "summary": "Final report creation failed.",
            "reason": str(e),
        }

        return {
            "current_step": "reporting_agent",
            "error": str(e),
            "logs": [new_log],
            "status": "failed",
        }